In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install rank_bm25 faiss-cpu sentence-transformers -q

import torch
import json
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm import tqdm
import faiss

DRIVE_BASE = "/content/drive/MyDrive/medrag"
DATA_PATH  = f"{DRIVE_BASE}/pubmedqa_filtered.json"
OUTPUT_DIR = f"{DRIVE_BASE}/biomistral_trial1_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Corpus: {len(corpus)} samples")

In [ ]:
MODEL_ID = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
    use_safetensors=False
)
model = model.to("cuda")
model.eval()
for param in model.parameters():
    param.requires_grad = False

n_layers   = model.config.num_hidden_layers
n_heads    = model.config.num_attention_heads
hidden_size = model.config.hidden_size
vocab_size  = model.config.vocab_size

print(f"Model: {n_layers} layers, {n_heads} heads, hidden {hidden_size}")

In [ ]:
documents, doc_metadata = [], []
for sample in corpus:
    for abstract in sample["supporting_abstracts"]:
        documents.append(abstract)
        doc_metadata.append({
            "pubid":       sample["pubid"],
            "label":       sample["label"],
            "gold_answer": sample["gold_answer"]
        })

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

enc_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
doc_embeddings = enc_model.encode(
    documents, batch_size=32,
    show_progress_bar=True, convert_to_numpy=True
)
faiss.normalize_L2(doc_embeddings)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print(f"Retriever ready: {index.ntotal} documents indexed")

In [ ]:
def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.lower().split())
    top_k  = np.argsort(scores)[::-1][:k]
    return [{"abstract": documents[i], "score": scores[i]} for i in top_k]

def faiss_retrieve(query, k=3):
    qe = enc_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(qe)
    scores, indices = index.search(qe, k)
    return [{"abstract": documents[i], "score": float(s)}
            for s, i in zip(scores[0], indices[0])]

def hybrid_retrieve(query, k=3, rrf_k=60):
    combined = {}
    for rank, r in enumerate(bm25_retrieve(query, k=k*2)):
        combined[r["abstract"]] = {"meta": r, "score": 1/(rrf_k+rank+1)}
    for rank, r in enumerate(faiss_retrieve(query, k=k*2)):
        key = r["abstract"]
        if key in combined:
            combined[key]["score"] += 1/(rrf_k+rank+1)
        else:
            combined[key] = {"meta": r, "score": 1/(rrf_k+rank+1)}
    return [v["meta"] for v in sorted(
        combined.values(), key=lambda x: x["score"], reverse=True)[:k]]

def build_rag_prompt(query, k=3):
    results = hybrid_retrieve(query, k=k)
    context = "\n\n".join(f"[{i+1}] {r['abstract']}"
                          for i, r in enumerate(results))
    return f"Context:\n{context}\n\nQuestion: {query}\nAnswer:", results

def get_context_end_idx(query, retrieved, tokenizer):
    """
    Returns the token index where context ends and Question: begins.
    Used to separate context attention from question attention.
    """
    context = "\n\n".join(f"[{i+1}] {r['abstract']}"
                          for i, r in enumerate(retrieved))
    context_prefix = f"Context:\n{context}\n\n"
    context_ids = tokenizer(
        context_prefix, return_tensors="pt", add_special_tokens=False
    )["input_ids"].shape[1]
    return context_ids

print("Retrieval functions ready.")

In [ ]:
N_SAMPLES   = len(corpus)
trial1_results = []

print(f"Running Trial 1 (attention) on {N_SAMPLES} samples...\n")

for idx, sample in enumerate(tqdm(corpus[:N_SAMPLES])):
    query = sample["query"]

    rag_prompt, retrieved = build_rag_prompt(query, k=3)
    inputs = tokenizer(
        rag_prompt, return_tensors="pt",
        truncation=True, max_length=512
    ).to("cuda")
    seq_len = inputs["input_ids"].shape[1]

    context_end = get_context_end_idx(query, retrieved, tokenizer)
    context_end = min(context_end, seq_len - 1)

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    sink_trajectory    = []
    context_trajectory = []

    for layer_idx in range(n_layers):
        attn = outputs.attentions[layer_idx].squeeze(0).float()
        last_token_attn = attn[:, -1, :]

        sink_pct = float(last_token_attn[:, 0].mean().item())

        if context_end > 1:
            ctx_pct = float(
                last_token_attn[:, 1:context_end].sum(dim=-1).mean().item()
            )
        else:
            ctx_pct = 0.0

        sink_trajectory.append(sink_pct)
        context_trajectory.append(ctx_pct)

    trial1_results.append({
        "idx":                idx,
        "pubid":              sample["pubid"],
        "query":              query[:60],
        "label":              sample["label"],
        "seq_len":            seq_len,
        "context_end":        context_end,
        "sink_trajectory":    sink_trajectory,
        "context_trajectory": context_trajectory,
        "mean_sink":          float(np.mean(sink_trajectory)),
        "mean_context_attn":  float(np.mean(context_trajectory)),
        "peak_sink_layer":    int(np.argmax(sink_trajectory)),
        "peak_context_layer": int(np.argmax(context_trajectory)),
    })

    if (idx + 1) % 50 == 0:
        ckpt_path = os.path.join(OUTPUT_DIR, f"trial1_checkpoint_{idx+1}.json")
        with open(ckpt_path, "w") as f:
            json.dump(trial1_results, f)
        torch.cuda.empty_cache()
        tqdm.write(f"Checkpoint saved at sample {idx+1}")

print(f"\nTrial 1 complete. {len(trial1_results)} samples processed.")

In [ ]:
print("TRIAL 1 SUMMARY BY LABEL\n")
for label in ["yes", "no"]:
    ls = [r for r in trial1_results if r["label"] == label]
    print(f"Label: {label} | n={len(ls)}")
    print(f"  Mean attention sink:       {np.mean([r['mean_sink'] for r in ls]):.4f}")
    print(f"  Mean peak sink layer:      {np.mean([r['peak_sink_layer'] for r in ls]):.1f}")
    print(f"  Mean context attention:    {np.mean([r['mean_context_attn'] for r in ls]):.4f}")
    print(f"  Mean peak context layer:   {np.mean([r['peak_context_layer'] for r in ls]):.1f}")
    print()

print("BioMedLM T1 reference:")
print("  Mean attention sink: 0.587 | Peak layer: 27.4")
print("  No label discrimination\n")

output_path = os.path.join(OUTPUT_DIR, "trial1_full_results.json")
with open(output_path, "w") as f:
    json.dump({
        "model_id":  MODEL_ID,
        "n_samples": len(trial1_results),
        "n_layers":  n_layers,
        "n_heads":   n_heads,
        "summary_by_label": {
            label: {
                "n":                   sum(1 for r in trial1_results if r["label"] == label),
                "mean_sink":           float(np.mean([r["mean_sink"]         for r in trial1_results if r["label"] == label])),
                "mean_peak_sink":      float(np.mean([r["peak_sink_layer"]   for r in trial1_results if r["label"] == label])),
                "mean_context_attn":   float(np.mean([r["mean_context_attn"] for r in trial1_results if r["label"] == label])),
                "mean_peak_ctx_layer": float(np.mean([r["peak_context_layer"] for r in trial1_results if r["label"] == label])),
            } for label in ["yes", "no"]
        },
        "samples": trial1_results
    }, f, indent=2)

for fname in os.listdir(OUTPUT_DIR):
    if fname.startswith("trial1_checkpoint"):
        os.remove(os.path.join(OUTPUT_DIR, fname))

print(f"Results saved to {output_path}")
print("05_biomistral_trial1_attention: complete")

In [ ]:
sink_by_layer = np.array([r["sink_trajectory"] for r in trial1_results])
ctx_by_layer  = np.array([r["context_trajectory"] for r in trial1_results])

mean_sink_per_layer = sink_by_layer.mean(axis=0)
mean_ctx_per_layer  = ctx_by_layer.mean(axis=0)

print(f"{'Layer':<8} {'Mean Sink':<16} {'Mean Context Attn'}")
print("-" * 42)
for i in range(n_layers):
    print(f"{i:<8} {mean_sink_per_layer[i]:<16.4f} {mean_ctx_per_layer[i]:.4f}")

print(f"\nSink: min={mean_sink_per_layer.min():.4f} (layer {mean_sink_per_layer.argmin()}), "
      f"max={mean_sink_per_layer.max():.4f} (layer {mean_sink_per_layer.argmax()})")
print(f"Context: min={mean_ctx_per_layer.min():.4f} (layer {mean_ctx_per_layer.argmin()}), "
      f"max={mean_ctx_per_layer.max():.4f} (layer {mean_ctx_per_layer.argmax()})")